In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
%cd "/content/drive/MyDrive/Colab Notebooks/NLP3_project"

In [ ]:
%ls

In [ ]:
!pip install simpletransformers transformers

In [ ]:
# prompt: read input from jsonl to dataframe

import pandas as pd

def jsonl_to_dataframe(file_path):
  """Reads a JSONL file and returns a Pandas DataFrame.

  Args:
    file_path: The path to the JSONL file.

  Returns:
    A Pandas DataFrame containing the data from the JSONL file.
  """
  try:
    df = pd.read_json(file_path, lines=True)
    return df
  except FileNotFoundError:
    print(f"Error: File not found at {file_path}")
    return None
  except pd.errors.ParserError:
    print(f"Error: Could not parse the JSONL file at {file_path}")
    return None
  except Exception as e:
    print(f"An unexpected error occurred: {e}")
    return None
# Example usage
# Replace 'your_file.jsonl' with the actual path to your JSONL file
df = jsonl_to_dataframe('data/train.jsonl')
df.head()


                                   uuid              postId  \
0  0af11f6b-c889-4520-9372-66ba25cb7657              532quh   
1  b1a1f63d-8853-4a11-89e8-6b2952a393ec  411701128456593408   
2  008b7b19-0445-4e16-8f9e-075b73f80ca4  380537005123190784   
3  31ecf93c-3e21-4c80-949b-aa549a046b93  844567852531286016   
4  31b108a3-c828-421a-a4b9-cf651e9ac859  814186311573766144   

                                            postText postPlatform  \
0  [Wes Welker Wanted Dinner With Tom Brady, But ...       reddit   
1   [NASA sets date for full recovery of ozone hole]      Twitter   
2  [This is what makes employees happy -- and it'...      Twitter   
3  [Passion is overrated — 7 work habits you need...      Twitter   
4  [The perfect way to cook rice so that it's per...      Twitter   

                                    targetParagraphs  \
0  [It’ll be just like old times this weekend for...   
1  [2070 is shaping up to be a great year for Mot...   
2  [Despite common belief, money isn't

In [ ]:
train_df_raw = jsonl_to_dataframe('data/train.jsonl')
val_df_raw = jsonl_to_dataframe('data/val.jsonl')
test_df_raw = jsonl_to_dataframe('data/test.jsonl')

In [ ]:
def load_input(df):
    if type(df) != pd.DataFrame:
        df = pd.read_json(df, lines=True)

    ret = []
    for _, i in df.iterrows():
        ret += [{'text': ' '.join(i['postText']) + ' - ' + i['targetTitle'] + ' ' + ' '.join(i['targetParagraphs']), 'uuid': i['uuid'],"labels":i["tags"][0]}]

    return pd.DataFrame(ret)

In [ ]:
train_df = load_input(train_df_raw)
val_df = load_input(val_df_raw)
test_df = load_input(test_df_raw)

In [ ]:
train_df.to_csv('data/train.csv', index=False)
val_df.to_csv('data/val.csv', index=False)
test_df.to_csv('data/test.csv', index=False)

In [ ]:
# prompt: from train_df extract only text, labels

train_df_model = train_df[['text', 'labels']]
val_df_model = val_df[['text', 'labels']]
test_df_model = test_df[['text', 'labels']]


In [ ]:
train_df

                                                   text  \
0     Wes Welker Wanted Dinner With Tom Brady, But P...   
1     NASA sets date for full recovery of ozone hole...   
2     This is what makes employees happy -- and it's...   
3     Passion is overrated — 7 work habits you need ...   
4     The perfect way to cook rice so that it's perf...   
...                                                 ...   
3195  Has Facebook's video explosion completely shak...   
3196  Cop Is Eating At A Chili's When Teen Hands Him...   
3197  5 popular myths about visible signs of aging t...   
3198  You need to see this Twitter account that pred...   
3199  GOP congressman comes out for gay marriage - P...   

                                      uuid   labels  
0     0af11f6b-c889-4520-9372-66ba25cb7657  passage  
1     b1a1f63d-8853-4a11-89e8-6b2952a393ec   phrase  
2     008b7b19-0445-4e16-8f9e-075b73f80ca4   phrase  
3     31ecf93c-3e21-4c80-949b-aa549a046b93    multi  
4     31b108a3-c828-4

In [ ]:
from sklearn.metrics import balanced_accuracy_score

def balanced_acc(labels, preds):
    return {
        "balanced_accuracy": balanced_accuracy_score(labels, preds)
    }

In [ ]:
from simpletransformers.classification import ClassificationModel,ClassificationArgs
import torch
cuda_available = torch.cuda.is_available()
model_args = ClassificationArgs()
model_args.labels_list = list(train_df['labels'].unique())
model_args.num_train_epochs = 10
model_args.overwrite_output_dir = True
model = ClassificationModel('roberta', 'roberta-base')
uuids = list(train_df['uuid'])
texts = list(train_df['text'])
predictions = model.predict(texts)[1]

In [ ]:
predictions

array([[0.15686035, 0.10540771],
       [0.1315918 , 0.11376953],
       [0.15124512, 0.10791016],
       ...,
       [0.15270996, 0.12548828],
       [0.15820312, 0.10998535],
       [0.14428711, 0.10614014]])

In [ ]:
train_df_model

                                                   text   labels
0     Wes Welker Wanted Dinner With Tom Brady, But P...  passage
1     NASA sets date for full recovery of ozone hole...   phrase
2     This is what makes employees happy -- and it's...   phrase
3     Passion is overrated — 7 work habits you need ...    multi
4     The perfect way to cook rice so that it's perf...   phrase
...                                                 ...      ...
3195  Has Facebook's video explosion completely shak...  passage
3196  Cop Is Eating At A Chili's When Teen Hands Him...  passage
3197  5 popular myths about visible signs of aging t...    multi
3198  You need to see this Twitter account that pred...   phrase
3199  GOP congressman comes out for gay marriage - P...   phrase

[3200 rows x 2 columns]

#HuggingFace

In [ ]:
import pandas as pd
from sklearn.preprocessing import LabelEncoder
from datasets import Dataset

# Load data
train_df = pd.read_csv("data/train.csv")  # or however you load it

# Encode labels
le = LabelEncoder()
train_df["label"] = le.fit_transform(train_df["labels"])  # tags → 0/1/2

# Convert to Hugging Face Dataset
dataset = Dataset.from_pandas(train_df[["text", "label", "uuid"]])

val_df = pd.read_csv("data/val.csv")
val_df["label"] = le.transform(val_df["labels"])
val_dataset = Dataset.from_pandas(val_df[["text", "label", "uuid"]])

test_df = pd.read_csv("data/test.csv")
test_df["label"] = le.transform(test_df["labels"])
test_dataset = Dataset.from_pandas(test_df[["text", "label", "uuid"]])


In [ ]:
!pip install evaluate

In [ ]:
import numpy as np
import pandas as pd
from sklearn.preprocessing import LabelEncoder
from datasets import Dataset
from transformers import (
    AutoTokenizer,
    AutoModelForSequenceClassification,
    Trainer,
    TrainingArguments
)
import evaluate
from sklearn.metrics import balanced_accuracy_score


from sklearn.metrics import balanced_accuracy_score, classification_report

class TextClassifierHF:
    def __init__(self, model_name="roberta-base", label_encoder=None, max_length=256):
        self.model_name = model_name
        self.tokenizer = AutoTokenizer.from_pretrained(model_name)
        self.max_length = max_length
        self.label_encoder = label_encoder or LabelEncoder()
        self.model = None
        self.trainer = None

    def tokenize_dataset(self, dataset):
        def tokenize(batch):
            return self.tokenizer(batch["text"], padding="max_length", truncation=True, max_length=self.max_length)
        return dataset.map(tokenize, batched=True)

    def setup(self, train_dataset, val_dataset, test_dataset):
        self.dataset = {
            "train": self.tokenize_dataset(train_dataset),
            "val": self.tokenize_dataset(val_dataset),
            "test": self.tokenize_dataset(test_dataset)
        }

    def compute_metrics(self, eval_pred):
        logits, labels = eval_pred
        preds = np.argmax(logits, axis=1)
        return {"balanced_accuracy": balanced_accuracy_score(labels, preds)}

    def train(self, output_dir="./results", epochs=5, batch_size=8):
        num_labels = len(np.unique(self.dataset["train"]["label"]))
        self.model = AutoModelForSequenceClassification.from_pretrained(
            self.model_name,
            num_labels=num_labels
        )

        args = TrainingArguments(
            output_dir=output_dir,
            evaluation_strategy="epoch",
            save_strategy="epoch",
            logging_dir=f"{output_dir}/logs",
            num_train_epochs=epochs,
            per_device_train_batch_size=batch_size,
            per_device_eval_batch_size=batch_size,
            load_best_model_at_end=True,
            metric_for_best_model="balanced_accuracy",
            greater_is_better=True,
            save_total_limit=1
        )

        self.trainer = Trainer(
            model=self.model,
            args=args,
            train_dataset=self.dataset["train"],
            eval_dataset=self.dataset["val"],
            tokenizer=self.tokenizer,
            compute_metrics=self.compute_metrics
        )

        self.trainer.train()

    def predict(self, split="test"):
        preds = self.trainer.predict(self.dataset[split])
        pred_labels = np.argmax(preds.predictions, axis=1)
        decoded_preds = self.label_encoder.inverse_transform(pred_labels)

        true_labels = self.label_encoder.inverse_transform(self.dataset[split]["label"])

        return pd.DataFrame({
            "uuid": self.dataset[split]["uuid"],
            "text": self.dataset[split]["text"],
            "true_tag": true_labels,
            "predicted_tag": decoded_preds
        })

    def evaluate(self, split="test", verbose=True):
        """Evaluate and print performance metrics like balanced accuracy and classification report."""
        preds = self.trainer.predict(self.dataset[split])
        pred_labels = np.argmax(preds.predictions, axis=1)
        true_labels = self.dataset[split]["label"]

        bal_acc = balanced_accuracy_score(true_labels, pred_labels)
        report = classification_report(true_labels, pred_labels, target_names=self.label_encoder.classes_)

        if verbose:
            print(f"Balanced Accuracy ({split}): {bal_acc:.4f}")
            print(f"\nClassification Report ({split}):\n{report}")

        return {
            "balanced_accuracy": bal_acc,
            "classification_report": report
        }



In [ ]:
# Load your data
import pandas as pd
from datasets import Dataset
from sklearn.preprocessing import LabelEncoder

train_df = pd.read_csv("data/train.csv")
val_df = pd.read_csv("data/val.csv")
test_df = pd.read_csv("data/test.csv")
model_name = "roberta-base"

# Encode labels
le = LabelEncoder()
train_df["label"] = le.fit_transform(train_df["labels"])
val_df["label"] = le.transform(val_df["labels"])
test_df["label"] = le.transform(test_df["labels"])

# Convert to Hugging Face Dataset format
train_dataset = Dataset.from_pandas(train_df[["text", "label", "uuid"]])
val_dataset = Dataset.from_pandas(val_df[["text", "label", "uuid"]])
test_dataset = Dataset.from_pandas(test_df[["text", "label", "uuid"]])
import torch
import gc

# Use the classifier
def end2endHF(model_name):
  print(f"="*50)
  print(f"Working on {model_name}")
  classifier = TextClassifierHF(model_name=model_name, label_encoder=le)
  classifier.setup(train_dataset, val_dataset, test_dataset)
  classifier.train()

  # Make predictions
  results_df = classifier.predict(split="test")
  print(results_df.head())
  safe_name = model_name.replace("/", "-")
  results_df.to_csv(f"{safe_name}_huggingface_predictions.csv", index=False)
  print(classifier.evaluate(split="test"))  # Or split="val"
  torch.cuda.empty_cache()
  gc.collect()
  print(f"="*50)

model_lists = ["roberta-base","bert-base-uncased","microsoft/deberta-v3-base","distilbert-base-uncased","bert-large-uncased","roberta-large","distilroberta-base"]
for model_name in model_lists:
  end2endHF(model_name)

Working on roberta-base


Some weights of RobertaForSequenceClassification were not initialized from the model checkpoint at roberta-base and are newly initialized: ['classifier.dense.bias', 'classifier.dense.weight', 'classifier.out_proj.bias', 'classifier.out_proj.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.
/usr/local/lib/python3.11/dist-packages/transformers/training_args.py:1611: FutureWarning: `evaluation_strategy` is deprecated and will be removed in version 4.46 of 🤗 Transformers. Use `eval_strategy` instead
  warnings.warn(
<ipython-input-20-d33596939ce0>:64: FutureWarning: `tokenizer` is deprecated and will be removed in version 5.0.0 for `Trainer.__init__`. Use `processing_class` instead.
  self.trainer = Trainer(
wandb: WARNING The `run_name` is currently set to the same value as `TrainingArguments.output_dir`. If this was not intended, please specify a different run name by setting the `TrainingArguments.run_name` parameter.

 ··········


Epoch,Training Loss,Validation Loss,Balanced Accuracy
1,No log,1.043504,0.333333
2,1.050600,1.042542,0.333333
3,1.043700,1.040696,0.333333
4,1.039000,1.038594,0.333333
5,1.040200,1.038695,0.333333


/usr/local/lib/python3.11/dist-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
/usr/local/lib/python3.11/dist-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
/usr/local/lib/python3.11/dist-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))


Balanced Accuracy (test): 0.3333

Classification Report (test):
              precision    recall  f1-score   support

       multi       0.00      0.00      0.00       174
     passage       0.00      0.00      0.00       403
      phrase       0.42      1.00      0.59       423

    accuracy                           0.42      1000
   macro avg       0.14      0.33      0.20      1000
weighted avg       0.18      0.42      0.25      1000

{'balanced_accuracy': np.float64(0.3333333333333333), 'classification_report': '              precision    recall  f1-score   support\n\n       multi       0.00      0.00      0.00       174\n     passage       0.00      0.00      0.00       403\n      phrase       0.42      1.00      0.59       423\n\n    accuracy                           0.42      1000\n   macro avg       0.14      0.33      0.20      1000\nweighted avg       0.18      0.42      0.25      1000\n'}
Working on bert-base-uncased


... (23 outputs left out)

Xet Storage is enabled for this repo, but the 'hf_xet' package is not installed. Falling back to regular HTTP download. For better performance, install the package with: `pip install huggingface_hub[hf_xet]` or `pip install hf_xet`


Some weights of RobertaForSequenceClassification were not initialized from the model checkpoint at distilroberta-base and are newly initialized: ['classifier.dense.bias', 'classifier.dense.weight', 'classifier.out_proj.bias', 'classifier.out_proj.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.
/usr/local/lib/python3.11/dist-packages/transformers/training_args.py:1611: FutureWarning: `evaluation_strategy` is deprecated and will be removed in version 4.46 of 🤗 Transformers. Use `eval_strategy` instead
  warnings.warn(
<ipython-input-20-d33596939ce0>:64: FutureWarning: `tokenizer` is deprecated and will be removed in version 5.0.0 for `Trainer.__init__`. Use `processing_class` instead.
  self.trainer = Trainer(


Epoch,Training Loss,Validation Loss,Balanced Accuracy
1,No log,0.840923,0.602328
2,0.920500,0.726705,0.660491
3,0.663900,0.858276,0.674689
4,0.390900,1.176108,0.686131
5,0.193300,1.526148,0.687205


Balanced Accuracy (test): 0.6648

Classification Report (test):
              precision    recall  f1-score   support

       multi       0.70      0.57      0.63       174
     passage       0.69      0.70      0.70       403
      phrase       0.68      0.73      0.70       423

    accuracy                           0.69      1000
   macro avg       0.69      0.66      0.68      1000
weighted avg       0.69      0.69      0.69      1000

{'balanced_accuracy': np.float64(0.6648285665988551), 'classification_report': '              precision    recall  f1-score   support\n\n       multi       0.70      0.57      0.63       174\n     passage       0.69      0.70      0.70       403\n      phrase       0.68      0.73      0.70       423\n\n    accuracy                           0.69      1000\n   macro avg       0.69      0.66      0.68      1000\nweighted avg       0.69      0.69      0.69      1000\n'}


In [ ]:

model_lists = ["roberta-base","roberta-large","bert-large-uncased"]
for model_name in model_lists:
  end2endHF(model_name)

Working on roberta-base


Some weights of RobertaForSequenceClassification were not initialized from the model checkpoint at roberta-base and are newly initialized: ['classifier.dense.bias', 'classifier.dense.weight', 'classifier.out_proj.bias', 'classifier.out_proj.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.
/usr/local/lib/python3.11/dist-packages/transformers/training_args.py:1611: FutureWarning: `evaluation_strategy` is deprecated and will be removed in version 4.46 of 🤗 Transformers. Use `eval_strategy` instead
  warnings.warn(
<ipython-input-20-d33596939ce0>:64: FutureWarning: `tokenizer` is deprecated and will be removed in version 5.0.0 for `Trainer.__init__`. Use `processing_class` instead.
  self.trainer = Trainer(


Epoch,Training Loss,Validation Loss,Balanced Accuracy
1,No log,1.035316,0.339331
2,0.972100,0.759084,0.639604
3,0.773200,0.728464,0.673738
4,0.554000,0.861163,0.702962
5,0.343700,1.084180,0.708980


Balanced Accuracy (test): 0.6951

Classification Report (test):
              precision    recall  f1-score   support

       multi       0.67      0.61      0.64       174
     passage       0.69      0.77      0.73       403
      phrase       0.76      0.70      0.73       423

    accuracy                           0.71      1000
   macro avg       0.71      0.70      0.70      1000
weighted avg       0.72      0.71      0.71      1000

{'balanced_accuracy': np.float64(0.6951330956253683), 'classification_report': '              precision    recall  f1-score   support\n\n       multi       0.67      0.61      0.64       174\n     passage       0.69      0.77      0.73       403\n      phrase       0.76      0.70      0.73       423\n\n    accuracy                           0.71      1000\n   macro avg       0.71      0.70      0.70      1000\nweighted avg       0.72      0.71      0.71      1000\n'}
Working on roberta-large


Some weights of RobertaForSequenceClassification were not initialized from the model checkpoint at roberta-large and are newly initialized: ['classifier.dense.bias', 'classifier.dense.weight', 'classifier.out_proj.bias', 'classifier.out_proj.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.
/usr/local/lib/python3.11/dist-packages/transformers/training_args.py:1611: FutureWarning: `evaluation_strategy` is deprecated and will be removed in version 4.46 of 🤗 Transformers. Use `eval_strategy` instead
  warnings.warn(
<ipython-input-20-d33596939ce0>:64: FutureWarning: `tokenizer` is deprecated and will be removed in version 5.0.0 for `Trainer.__init__`. Use `processing_class` instead.
  self.trainer = Trainer(


Epoch,Training Loss,Validation Loss,Balanced Accuracy
1,No log,1.044857,0.333333
2,1.064100,1.038812,0.333333
3,1.051600,1.039448,0.333333
4,1.043300,1.039674,0.333333
5,1.042900,1.038611,0.333333


... (2 outputs left out)

Some weights of BertForSequenceClassification were not initialized from the model checkpoint at bert-large-uncased and are newly initialized: ['classifier.bias', 'classifier.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.
/usr/local/lib/python3.11/dist-packages/transformers/training_args.py:1611: FutureWarning: `evaluation_strategy` is deprecated and will be removed in version 4.46 of 🤗 Transformers. Use `eval_strategy` instead
  warnings.warn(
<ipython-input-20-d33596939ce0>:64: FutureWarning: `tokenizer` is deprecated and will be removed in version 5.0.0 for `Trainer.__init__`. Use `processing_class` instead.
  self.trainer = Trainer(


Epoch,Training Loss,Validation Loss,Balanced Accuracy
1,No log,1.043727,0.333333
2,1.068000,1.042296,0.333333
3,1.055100,1.040517,0.333333
4,1.055500,1.038834,0.333333
5,1.050000,1.038715,0.333333


/usr/local/lib/python3.11/dist-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
/usr/local/lib/python3.11/dist-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
/usr/local/lib/python3.11/dist-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))


Balanced Accuracy (test): 0.3333

Classification Report (test):
              precision    recall  f1-score   support

       multi       0.00      0.00      0.00       174
     passage       0.00      0.00      0.00       403
      phrase       0.42      1.00      0.59       423

    accuracy                           0.42      1000
   macro avg       0.14      0.33      0.20      1000
weighted avg       0.18      0.42      0.25      1000

{'balanced_accuracy': np.float64(0.3333333333333333), 'classification_report': '              precision    recall  f1-score   support\n\n       multi       0.00      0.00      0.00       174\n     passage       0.00      0.00      0.00       403\n      phrase       0.42      1.00      0.59       423\n\n    accuracy                           0.42      1000\n   macro avg       0.14      0.33      0.20      1000\nweighted avg       0.18      0.42      0.25      1000\n'}


#SimpleTransformer

In [ ]:
from simpletransformers.classification import ClassificationModel

model = ClassificationModel(
    "roberta", "roberta-base",
    num_labels=3,
    use_cuda=torch.cuda.is_available(),
    args={
        "overwrite_output_dir": True,
        "reprocess_input_data": True,
        "evaluate_during_training": True,
        "num_train_epochs": 3,
        "train_batch_size": 8,
        "eval_batch_size": 8,
        "labels_list": ["passage","phrase","multi"],
        "evaluate_during_training_steps": 5,
        "save_eval_checkpoints": False,
        "save_model_every_epoch": False,
        "save_steps": -1,
        "fp16": False,
        "logging_steps": 5,
        "metric_for_best_model": "balanced_accuracy",
        "early_stopping_metric": "balanced_accuracy",
        "early_stopping_metric_minimize": False,
        "early_stopping_patience": 2
    }
)


In [ ]:
model = ClassificationModel(
    "roberta", "roberta-base",
    num_labels=3,args={"labels_list": ["passage","phrase","multi"],"overwrite_output_dir": True})


In [ ]:
assert train_df_model["text"].notnull().all()
assert train_df_model["labels"].notnull().all()

In [ ]:
model.train_model(train_df_model)

In [ ]:
model.train_model(train_df, eval_df=val_df, output_dir="outputs",
                  eval_fn=balanced_acc)

In [ ]:
preds, model_outputs = model.predict(test_df)
print("Predictions:", preds)

In [ ]:
!pip install transformers datasets evaluate scikit-learn